In [6]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score
from transformers import pipeline

In [ ]:
reviews_df = pd.read_csv("final_yelp_dataset.csv")
reviews_df.head(5)

# Takes a random sample for testing
reviews_sample = reviews_df.sample(n = 5000, random_state = 42)

# Balanced dataset (capped at 5000 per cuisine-tier group)
sampled_df = (
    reviews_df
    .groupby(['cuisine', 'tier'], group_keys=False)
    .apply(lambda x: x.sample(min(len(x), 5000), random_state=42))
    .reset_index(drop=True)
)

# Create working dataframe 
df = sampled_df.copy()

# Create binary labels (1 = positive, 0 = negative)
df['label'] = df['individual_stars'].apply(lambda x: 1 if x>=4 else 0
)

X = df['text']
y = df['label']

# Sets up 5-fold cross-validation 
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Stores the evaluation metrics
acc_scores = []
f1_scores = []
auc_scores = []


/var/folders/2t/h_cypqr94zx0kjmx1phwr9h40000gn/T/ipykernel_11740/1690164005.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 5000), random_state=42))


In [ ]:
# Load RoBERTa sentiment model
roberta_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=0
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 15664.05it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Removes the neutral reviews (3-star)
labeled_df = sampled_df[sampled_df['individual_stars'] != 3].copy()

# Creates binary labels (1 = positive, 0 = negative)
labeled_df['true_label'] = (labeled_df['individual_stars'] >= 4).astype(int)

# Check size of labeeled dataset
print(f"Labeled subset: {len(labeled_df)} reviews")

Labeled subset: 65727 reviews


In [ ]:
# Tests different max_length values
print("\n── Optimizing max_length ──")

for max_len in [64, 128, 256, 512]:
    preds = roberta_model(
        labeled_df['text'].tolist()[:500],
        truncation=True,
        max_length=max_len,
        batch_size=128
    )

    # Converts predictions to binary labels
    pred_labels = [
        1 if r['label'] in ['positive', 'neutral'] else 0
        for r in preds
    ]

    # Get true labels
    true_labels = labeled_df['true_label'].values[:500]

    # Compute evaluation metrics
    f1 = f1_score(true_labels, pred_labels)
    acc = accuracy_score(true_labels, pred_labels)

    print(f"  max_length={max_len:4d}:  F1={f1:.4f},  Accuracy={acc:.4f}")


── Optimizing max_length ──
  max_length=  64:  F1=0.9049,  Accuracy=0.9020
  max_length= 128:  F1=0.9234,  Accuracy=0.9220
  max_length= 256:  F1=0.9270,  Accuracy=0.9260
  max_length= 512:  F1=0.9291,  Accuracy=0.9280


In [ ]:
# Optimize classification threshold using cross validation
print("\n── Optimizing classification threshold (5-fold CV) ──")

# Takes a smaller sample for faster computation
sample_df = labeled_df.sample(1000, random_state=42).reset_index(drop=True)

# Gets model predictions
raw_preds = roberta_model(
    sample_df['text'].tolist(),
    truncation=True,
    max_length=128,
    batch_size=128
)

# Converts to positive probability
sample_df['raw_positive_score'] = [
    r['score'] if r['label'] == 'positive' else 1 - r['score']
    for r in raw_preds
]

# Sets up 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
thresholds = np.arange(0.3, 0.8, 0.05)

threshold_f1s = {t: [] for t in thresholds}

# Evaluates each threshold
for _, val_idx in kf.split(sample_df):
    val = sample_df.iloc[val_idx]

    for thresh in thresholds:
        preds = (val['raw_positive_score'] >= thresh).astype(int)
        f1 = f1_score(val['true_label'], preds)
        threshold_f1s[thresh].append(f1)

print(f"\n  {'Threshold':>10}  {'Mean F1':>10}  {'Std':>8}")

best_thresh, best_f1 = 0.5, 0

# Finds the best threshold
for thresh, f1s in threshold_f1s.items():
    mean_f1 = np.mean(f1s)
    std_f1 = np.std(f1s)

    print(f"  {thresh:>10.2f}  {mean_f1:>10.4f}  {std_f1:>8.4f}")

    if mean_f1 > best_f1:
        best_f1 = mean_f1
        best_thresh = thresh

print(f"\nBest threshold: {best_thresh:.2f} (F1={best_f1:.4f})")


── Optimizing classification threshold (5-fold CV) ──

   Threshold     Mean F1       Std
        0.30      0.9632    0.0078
        0.35      0.9668    0.0050
        0.40      0.9705    0.0052
        0.45      0.9691    0.0041
        0.50      0.9637    0.0059
        0.55      0.9638    0.0095
        0.60      0.9583    0.0126
        0.65      0.9549    0.0130
        0.70      0.9535    0.0129
        0.75      0.9505    0.0141

Best threshold: 0.40 (F1=0.9705)
